# Test Signature & Cachet PDF

Ce notebook teste l'ajout d'une signature et d'un cachet sur un PDF avec `pymupdf`.

**Structure attendue :**
```
rd/signature/
  assets/
    signature.png   ← image de la signature (fond transparent de préférence)
    cachet.png      ← image du cachet (fond transparent de préférence)
  input/
    document.pdf    ← le PDF à signer
  output/           ← PDF signé généré ici
```

In [1]:
# Installation si nécessaire
!pip install pymupdf pillow

   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -------------------------------------- - 6.8/7.0 MB 41.7 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 36.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import fitz  # pymupdf
from pathlib import Path
from PIL import Image
import os

# --- Chemins ---
BASE = Path(".")  # dossier rd/signature/
PDF_INPUT  = BASE / "input" / "CPS 27-2025.pdf"
PDF_OUTPUT = BASE / "output" / "CPS 27-2025.pdf"
SIGNATURE  = BASE / "assets" / "signature.jpg"
CACHET     = BASE / "assets" / "cachet.png"

# Créer les dossiers si absents
for d in [BASE/"input", BASE/"output", BASE/"assets"]:
    d.mkdir(parents=True, exist_ok=True)

print("PDF input   :", PDF_INPUT.exists())
print("Signature   :", SIGNATURE.exists())
print("Cachet      :", CACHET.exists())

PDF input   : True
Signature   : True
Cachet      : True


In [3]:
# --- Paramètres de placement ---

# Signature : bas droite de chaque page
# Rect(x0, y0, x1, y1) en points PDF (1pt = 1/72 pouce)
SIGNATURE_WIDTH  = 120   # largeur en points
SIGNATURE_HEIGHT = 50    # hauteur en points
SIGNATURE_MARGIN_X = 50  # marge depuis bord droit
SIGNATURE_MARGIN_Y = 40  # marge depuis bas de page

# Cachet : dernière page, bas gauche
CACHET_WIDTH  = 100
CACHET_HEIGHT = 100
CACHET_MARGIN_X = 50
CACHET_MARGIN_Y = 40

In [8]:
def ajouter_signature_cachet(
    pdf_path: Path,
    output_path: Path,
    signature_path: Path,
    cachet_path: Path,
    sig_w=120, sig_h=50, sig_mx=50, sig_my=40,
    cac_w=100, cac_h=100, cac_mx=50, cac_my=40,
):
    doc = fitz.open(str(pdf_path))
    n_pages = len(doc)

    sig_bytes = open(signature_path, "rb").read()
    cac_bytes = open(cachet_path, "rb").read()

    for i, page in enumerate(doc):
        pw = page.rect.width
        ph = page.rect.height

        # --- Signature bas droite ---
        sig_rect = fitz.Rect(
            pw - sig_mx - sig_w,
            ph - sig_my - sig_h,
            pw - sig_mx,
            ph - sig_my,
        )
        page.insert_image(sig_rect, stream=sig_bytes, keep_proportion=True)

        # --- Cachet bas gauche — uniquement dernière page ---
        if i == n_pages - 1:
            cac_rect = fitz.Rect(
                cac_mx,
                ph - cac_my - cac_h,
                cac_mx + cac_w,
                ph - cac_my,
            )
            page.insert_image(cac_rect, stream=cac_bytes, keep_proportion=True)

    doc.save(str(output_path))
    doc.close()
    print(f"✓ PDF signé sauvegardé : {output_path}")
    print(f"  {n_pages} page(s) traitée(s)")


# --- Lancement ---
ajouter_signature_cachet(
    pdf_path=PDF_INPUT,
    output_path=PDF_OUTPUT,
    signature_path=SIGNATURE,
    cachet_path=CACHET,
    sig_w=SIGNATURE_WIDTH, sig_h=SIGNATURE_HEIGHT,
    sig_mx=SIGNATURE_MARGIN_X, sig_my=SIGNATURE_MARGIN_Y,
    cac_w=CACHET_WIDTH, cac_h=CACHET_HEIGHT,
    cac_mx=CACHET_MARGIN_X, cac_my=CACHET_MARGIN_Y,
)

✓ PDF signé sauvegardé : output\CPS 27-2025.pdf
  18 page(s) traitée(s)


In [ ]:
# --- Aperçu de la dernière page du PDF signé ---
import matplotlib.pyplot as plt
import numpy as np

doc = fitz.open(str(PDF_OUTPUT))
last_page = doc[-1]
mat = fitz.Matrix(1.5, 1.5)  # zoom x1.5
pix = last_page.get_pixmap(matrix=mat)
img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

plt.figure(figsize=(10, 14))
plt.imshow(img)
plt.axis("off")
plt.title(f"Dernière page (page {len(doc)}) — signature + cachet")
plt.tight_layout()
plt.show()
doc.close()